# Milestone 2: Feature Engineering

**Dataset:** House Sales in King County, USA (same as Milestone 1)

**Goal:** Change the columns into numbers that a Machine Learning model can use. I also create new features.

**Target variable:** `price`. I keep it as it is. I do not scale it.

## Step 1: Import libraries and load the data

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder

df = pd.read_csv('../data/kc_house_data.csv')
print(df.shape)
df.head()

## Step 2: Create new features

I make new columns from the old ones. I do this first because some of the new columns are text. I will turn them into numbers in the next steps.

### New feature 1: `house_age`

`house_age = year of sale - year built`

**Why this feature is helpful:**
- A model cannot use `yr_built` well. The year 1990 is only a number to it.
- Age has a clear meaning. A newer house usually costs more. An older house usually costs less.
- Age also puts all sale years on the same scale.

### New feature 2: `renovation_status`

This says if the house was renovated or not. In the data, `yr_renovated` is 0 when there was no renovation. A model can be confused by this 0 because it looks like a year. A clear label is easier to understand.

### New feature 3: `season`

This is the season of the sale (Winter, Spring, Summer or Autumn). Prices can change during the year. A season is easier to use than a date written as text.

### New feature 4: `total_rooms`

`total_rooms = bedrooms + bathrooms`

This shows the overall size of the house in one number.

### Why I did not make `price_per_sqft`

The task gives `price_per_sqft` as an example. It is made using `price`, and `price` is the value I want to predict. The model would see part of the answer. This is called **data leakage**. It would give very good results that are not real. So I do not add it to the final dataset.

In [ ]:
# The date is text like '20141013T000000'. I change it to a real date.
df['date'] = pd.to_datetime(df['date'], format='%Y%m%dT%H%M%S')
df['sale_year'] = df['date'].dt.year
df['sale_month'] = df['date'].dt.month

# New feature 1: house age
df['house_age'] = df['sale_year'] - df['yr_built']

# New feature 2: renovation status (text label)
df['renovation_status'] = np.where(df['yr_renovated'] > 0, 'Renovated', 'Not Renovated')

# New feature 3: season (text label)
season_map = {12: 'Winter', 1: 'Winter', 2: 'Winter',
              3: 'Spring', 4: 'Spring', 5: 'Spring',
              6: 'Summer', 7: 'Summer', 8: 'Summer',
              9: 'Autumn', 10: 'Autumn', 11: 'Autumn'}
df['season'] = df['sale_month'].map(season_map)

# New feature 4: total rooms
df['total_rooms'] = df['bedrooms'] + df['bathrooms']

df[['sale_year', 'yr_built', 'house_age', 'renovation_status', 'season', 'total_rooms']].head()

A house can have a negative age if it was sold before it was built. This is rare. I check it here.

In [ ]:
print("Houses with a negative age:", (df['house_age'] < 0).sum())
df['house_age'] = df['house_age'].clip(lower=0)

## Step 3: Remove columns that are not useful

- `id` is only a label. It does not help the model.
- `date` is text or a date. I already took the year and month from it.
- `yr_renovated` is now covered by `renovation_status`.

In [ ]:
df = df.drop(columns=['id', 'date', 'yr_renovated'])
df.shape

## Step 4: Find the categorical columns

These are the columns with text labels. `zipcode` is a number, but it is really a place name. The zipcode 98100 is not "bigger" than 98000. So I treat it as a category and change it to text.

In [ ]:
df['zipcode'] = df['zipcode'].astype(str)

categorical_cols = df.select_dtypes(include=['object', 'string']).columns.tolist()
print("Categorical columns:", categorical_cols)

## Step 5: Apply encoding

I use two methods.

- **Label Encoding** is for a column with 2 values. `renovation_status` has only 2 values, so it becomes 0 and 1.
- **One-Hot Encoding** is for a column with no order. `season` and `zipcode` have no order, so each value gets its own 0/1 column.

In [ ]:
# Label Encoding for the binary column
le = LabelEncoder()
df['renovation_status'] = le.fit_transform(df['renovation_status'])
print("Label Encoder classes (0 and 1):", list(le.classes_))

# One-Hot Encoding for the other categories
df = pd.get_dummies(df, columns=['season', 'zipcode'], dtype=int)

print("Shape after encoding:", df.shape)

## Step 6: Find numerical columns with very different scales

Some columns are very big (like `sqft_lot`). Some are very small (like `floors`). A model can give too much weight to the big numbers.

In [ ]:
cols_to_check = ['bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors',
                 'grade', 'sqft_above', 'sqft_basement', 'yr_built',
                 'sqft_living15', 'sqft_lot15', 'house_age', 'total_rooms']
df[cols_to_check].describe().loc[['min', 'max', 'mean']]

## Step 7: Apply scaling

`StandardScaler` changes each column so the mean is 0 and the spread is 1.

**What I scale:** the number columns that have different sizes.

**What I do not scale:**
- `price`, because it is my target.
- The 0/1 columns (encoded columns, `waterfront`, `renovation_status`). They are already small.

Note: Later, when I split the data into train and test, I should fit the scaler on the train data only. Here I fit it on all the data to keep this milestone simple.

In [ ]:
cols_to_scale = ['bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors',
                 'view', 'condition', 'grade', 'sqft_above', 'sqft_basement',
                 'yr_built', 'lat', 'long', 'sqft_living15', 'sqft_lot15',
                 'sale_year', 'sale_month', 'house_age', 'total_rooms']

scaler = StandardScaler()
df[cols_to_scale] = scaler.fit_transform(df[cols_to_scale])

df[cols_to_scale].describe().loc[['mean', 'std']].round(2)

The mean is about 0 and the standard deviation is about 1 for every scaled column. The scaling worked.

## Step 8: Check that the data is ready for a model

I check three things:
1. All columns are numbers.
2. There are no missing values.
3. The target `price` is still in its original form.

In [ ]:
print("Non-number columns:", df.select_dtypes(exclude='number').columns.tolist())
print("Missing values:", df.isnull().sum().sum())
print("Final shape:", df.shape)
df['price'].describe()

## Step 9: Save the engineered dataset

In [ ]:
df.to_csv('../data/engineered_data.csv', index=False)
print("Saved to data/engineered_data.csv")

## Step 10: Save the scaler and column lists

My Streamlit app (Milestone 5) must change new data in the same way. So I save the scaler, the scaled column names, the final column order and the label classes.

In [ ]:
import joblib

preprocessing = {
    'scaler': scaler,
    'cols_to_scale': cols_to_scale,
    'feature_columns': [c for c in df.columns if c != 'price'],
    'renovation_classes': list(le.classes_),
}
joblib.dump(preprocessing, '../preprocessing.joblib')
print("Saved to preprocessing.joblib")

## Summary

- I made 4 new features: `house_age`, `renovation_status`, `season` and `total_rooms`.
- I changed all text columns to numbers. I used Label Encoding for 2 values and One-Hot Encoding for the rest.
- I scaled the number columns with `StandardScaler`.
- I did not use `price_per_sqft`, because it would leak the answer to the model.
- The final data is saved as `data/engineered_data.csv` and is ready for a model.
- I saved the scaler and column lists in `preprocessing.joblib` for the app.